In [1]:
import importlib

In [2]:


import pandas as pd
import data_loader
import user_profile as user_pf


In [3]:

# importlib.reload(user_pf)

In [4]:
# Change this to True if you are running on Google Colab
RUNNING_ON_COLAB = False

# Constants
USER_ID = 999999

In [5]:
if RUNNING_ON_COLAB:
    data_loader.mount_drive()

In [6]:
# Load the data
films_df = data_loader.load_movies()
ratings_df = data_loader.load_ratings()
credits_df = data_loader.load_credits()

/Users/Cathal/Rec-Genie/rec-sys/data_loader.py:16: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(dataset_path + "/movies_metadata.csv", usecols=['id', 'title', 'release_date', 'genres', 'popularity', 'vote_average', 'vote_count'])


In [7]:
credits_df.shape

(45476, 3)

In [8]:
# Drop the weird film entry
films_df = films_df.drop(35587)

In [9]:
import preprocessing as pre

In [10]:
importlib.reload(pre)

<module 'preprocessing' from '/Users/Cathal/Rec-Genie/rec-sys/preprocessing.py'>

In [11]:
# Data preprocessing
# One-hot encode the genres
ohe_films_df, genre_list_mlb = pre.one_hot_encode_genres(films_df)

In [12]:
ohe_films_df.columns

Index(['id', 'popularity', 'release_date', 'title', 'vote_average',
       'vote_count', 'Action', 'Adventure', 'Animation', 'Comedy', 'Crime',
       'Documentary', 'Drama', 'Family', 'Fantasy', 'Foreign', 'History',
       'Horror', 'Music', 'Mystery', 'Romance', 'Science Fiction', 'TV Movie',
       'Thriller', 'War', 'Western'],
      dtype='object')

In [13]:
# Gather info on directors and cast
credits_df = pre.condense_credits(credits_df)

In [14]:
print(type(ohe_films_df), type(credits_df))  # Debugging line

<class 'pandas.core.frame.DataFrame'> <class 'pandas.core.frame.DataFrame'>


In [15]:

# Tidy the noise and merge the credits' metadata with the films DataFrame
films_df = pre.data_tidying(ohe_films_df, credits_df)
print(films_df.columns)


Index(['id', 'popularity', 'release_date', 'title', 'vote_average',
       'vote_count', 'Action', 'Adventure', 'Animation', 'Comedy', 'Crime',
       'Documentary', 'Drama', 'Family', 'Fantasy', 'Foreign', 'History',
       'Horror', 'Music', 'Mystery', 'Romance', 'Science Fiction', 'TV Movie',
       'Thriller', 'War', 'Western', 'cast_info', 'director_info'],
      dtype='object')


In [16]:
# Generate the user profile
user_ratings_df = user_pf.load_user_ratings()
user_profile = user_pf.create_user_profile(USER_ID, films_df, user_ratings_df, genre_list_mlb)

In [17]:
# Append the user profile to the ratings DataFrame
ratings_df = pd.concat([ratings_df, user_ratings_df], ignore_index=True)

In [70]:
import hybrid_recommender as hyb
importlib.reload(hyb)


<module 'hybrid_recommender' from '/Users/Cathal/Rec-Genie/rec-sys/hybrid_recommender.py'>

In [45]:

# Generate recommendations
recommendations = hyb.hybrid_recommend(999999, user_profile, films_df, credits_df, ratings_df, genre_list_mlb)


User-User algorithm set up!


In [46]:
len(recommendations)

400

In [49]:
score_breakdown = hyb.score_breakdown(films_df, recommendations)

In [ ]:
score_breakdown.head(50)

In [24]:
# weights = {
#     'cast_ft_weight': 0.3,
#     'director_ft_weight': 0.3,
#     'genre_ft_weight': 0.15,
#     'user_user_weight': 1,
#     'content_weight': 0.7,
#     'collab_weight': 0.3
# }